<style>
/* FABRIC notebook  adaptive theme */
.fab-info    { background-color: #f0f7fb; border-left: 4px solid #1f6a8c; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-success { background-color: #e8f5e9; border-left: 4px solid #008e7a; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-warning { background-color: #fff8e1; border-left: 4px solid #ff8542; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-danger  { background-color: #fce4ec; border-left: 4px solid #b00020; padding: 12px 15px; margin: 15px 0; border-radius: 4px; }
.fab-footer  { background-color: #374955; color: white; padding: 15px 20px; margin: 20px 0; border-radius: 4px; text-align: center; }

@media (prefers-color-scheme: dark) {
  .fab-info    { background-color: #1a2a35; border-color: #5798bc; color: #d0e0eb; }
  .fab-success { background-color: #1a2e25; border-color: #00b89a; color: #c0e0d5; }
  .fab-warning { background-color: #2e2518; border-color: #ff9a5c; color: #e0d0b8; }
  .fab-danger  { background-color: #2e1a1e; border-color: #e53950; color: #e0c0c8; }
  .fab-footer  { background-color: #2a3a45; color: #b0c4d0; }
}
</style>

# FABnet IPv6 Network: Fully Automatic Configuration

<picture>
  <source srcset="../../images/fabric_logo_light.png" media="(prefers-color-scheme: dark)">
  <img src="../../images/fabric_logo.png" width="300" style="margin-bottom:10px;"/>
</picture>

<div class="fab-info">

**What this notebook does:** Creates a two-node experiment spanning two FABRIC sites connected via **FABnet IPv6** using the **fully automatic** approach. A single `add_fabnet()` call on each node handles NIC creation, network setup, IP assignment, and routing -- this is the simplest way to get IPv6 connectivity on FABRIC.

</div>

## Learning Objectives

<div class="fab-success">

After completing this notebook you will be able to:

1. Use `node.add_fabnet(net_type='IPv6')` to automatically connect a node to the FABnet IPv6 service
2. Understand how fully automatic mode creates interfaces, networks, and routes behind the scenes
3. Retrieve the auto-generated network name and query a node's IPv6 address
4. Verify IPv6 connectivity between nodes on different FABRIC sites

</div>

## Prerequisites

<div class="fab-warning">

Before running this notebook you **must**:

1. Run the [Configure Environment](../../../configure_and_validate/configure_and_validate.ipynb) notebook
2. Be familiar with creating basic slices -- see [Hello, FABRIC](../../hello_fabric/hello_fabric.ipynb)

**Tip -- Three configuration approaches:** FABRIC offers three ways to set up FABnet IPv6:

| Approach | Notebook | You create networks? | You assign IPs? |
|----------|----------|---------------------|-----------------|
| **Full Auto** (this notebook) | You are here | No (`add_fabnet()`) | No |
| **Auto** | [auto](create_l3network_fabnet_ipv6_auto.ipynb) | Yes | No (auto mode) |
| **Manual** | [manual](create_l3network_fabnet_ipv6_manual.ipynb) | Yes | Yes |

</div>

## Background: Fully Automatic FABnet IPv6

FABRIC provides Layer 3 IP networking services (FABnetv4 and FABnetv6) across every site. These act as a private internet connecting experiments over high-performance links.

With **fully automatic** mode, a single call to `node.add_fabnet()` does everything:

- Adds a NIC interface to the node
- Creates (or reuses) a FABnet network on the node's site
- Sets the interface to auto mode for IP configuration
- Configures routes during the post-boot phase

If multiple nodes reside on the **same** site, they are automatically connected to the same FABnet network.


The auto-generated network names follow the pattern `FABNET_IPv6_<site_name>`. This is the name you use to query the interface's IP address.

## What We're Building

In this notebook we will create two nodes on different sites connected via FABNetv6.

<img src="./figs/slice_topology.png" width="50%">


---

## Step 1: Import FABlib and Verify Configuration

In [ ]:
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

# Create a FABlib manager instance
fablib = fablib_manager()

# Display current configuration (tokens, keys, project info)
fablib.show_config();

## Step 2: Define Slice Parameters

We select two distinct sites. With full auto mode, we only need site names and node names -- no network or NIC names to manage.

In [ ]:
# Name for this experiment slice
slice_name = 'MySlice'

# Pick two distinct random FABRIC sites
[site1, site2] = fablib.get_random_sites(count=2)
print(f"Sites: {site1}, {site2}")

# Node names
node1_name = 'Node1'
node2_name = 'Node2'

## Step 3: Build and Submit the Slice

With fully automatic mode, the slice definition is remarkably simple. For each node, call `add_fabnet(net_type='IPv6')` and FABlib takes care of everything else.

<div class="fab-warning">

**Tip:** This is the recommended approach for anyone who does not need custom control over IP addresses and routes. It is the simplest way to connect your nodes to FABnet.

</div>

In [ ]:
# Create a new empty slice
slice = fablib.new_slice(name=slice_name)

# --- Node1: add to site1 and connect to FABnet IPv6 ---
node1 = slice.add_node(name=node1_name, site=site1)
# add_fabnet() creates a NIC, connects it to a FABnet IPv6 network,
# and configures the IP address and routes automatically
node1.add_fabnet(net_type='IPv6')

# --- Node2: add to site2 and connect to FABnet IPv6 ---
node2 = slice.add_node(name=node2_name, site=site2)
node2.add_fabnet(net_type='IPv6')

# Submit the slice -- blocks until provisioning is complete (~2-5 min)
slice.submit();

<div class="fab-success">

**What just happened?** FABlib created a NIC on each node, connected each to a FABnet IPv6 network at its site, assigned IPv6 addresses, and configured inter-site routes. The slice is ready to use immediately.

</div>

---

## Step 4: Run the Experiment

With fully automatic configuration, the slice is ready for experimentation as soon as it becomes active. We verify connectivity by pinging Node2 from Node1.

Note the auto-generated network name pattern: `FABNET_IPv6_<site_name>`. We use this to look up Node2's IPv6 address.

In [ ]:
# Retrieve the slice (useful if returning to this notebook later)
slice = fablib.get_slice(slice_name)

# Get node objects
node1 = slice.get_node(name=node1_name)
node2 = slice.get_node(name=node2_name)

# The auto-generated network name follows the pattern: FABNET_IPv6_<site_name>
# Use get_site() to construct the network name dynamically
node2_addr = node2.get_interface(network_name=f'FABNET_IPv6_{node2.get_site()}').get_ip_addr()

# Ping Node2 from Node1 across the FABRIC backbone
stdout, stderr = node1.execute(f'ping -c 5 {node2_addr}')

## Step 5: Delete the Slice

<div class="fab-danger">

**Important:** Always delete your slice when you are done. FABRIC is a shared resource -- leaving slices running prevents other researchers from using those resources.

</div>

In [ ]:
# Delete the slice and release all resources
slice.delete()

---

## Troubleshooting

| Problem | Possible Cause | Solution |
|---------|---------------|----------|
| Slice stuck in `Configuring` | A site may be busy or temporarily unavailable | Try different sites by re-running `get_random_sites()` |
| `ping` fails between nodes | Auto configuration may not have completed | SSH into the node and check `ip -6 addr show` and `ip -6 route list` |
| Cannot find network by name | Wrong network name pattern | The auto-generated name is `FABNET_IPv6_<SITENAME>` (all caps) |
| `get_ip_addr()` returns None | Post-boot config still running | Wait a minute and re-query the slice with `fablib.get_slice()` |
| `PDP Authorization check failed` | Project permissions issue | Contact your project lead or FABRIC support |

## FABlib API Reference

| Method | Description | Documentation |
|--------|-------------|---------------|
| `fablib.get_random_sites(count)` | Select distinct random sites | [get_random_sites](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.get_random_sites) |
| `node.add_fabnet(net_type)` | Add a FABnet connection (NIC + network + IP + routes) | [add_fabnet](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.add_fabnet) |
| `node.get_site()` | Get the site name where the node is located | [get_site](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.get_site) |
| `node.get_interface(network_name)` | Get the interface connected to a specific network | [get_interface](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.get_interface) |
| `iface.get_ip_addr()` | Get the IP address assigned to the interface | [get_ip_addr](https://fabric-fablib.readthedocs.io/en/latest/interface.html#fabrictestbed_extensions.fablib.interface.Interface.get_ip_addr) |
| `node.execute(command)` | Execute a shell command on a node | [execute](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.execute) |
| `slice.submit()` | Submit the slice for provisioning | [submit](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.submit) |
| `slice.delete()` | Delete the slice and release resources | [delete](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.delete) |

## What's Next?

| Topic | Notebook | What You'll Learn |
|-------|----------|-------------------|
| **FABnet IPv6 Auto** | [auto](create_l3network_fabnet_ipv6_auto.ipynb) | More control: create networks yourself, auto IP assignment |
| **FABnet IPv6 Manual** | [manual](create_l3network_fabnet_ipv6_manual.ipynb) | Full control over IP address assignment |
| **FABnet IPv4 Full Auto** | [ipv4_full_auto](../create_l3network_fabnet_ipv4/create_l3network_fabnet_ipv4_full_auto.ipynb) | Same approach with IPv4 addressing |
| **FABnet IPv6 Ext** | [ipv6_ext](../create_l3network_fabnet_ipv6ext_manual/create_l3network_fabnet_ipv6ext_manual.ipynb) | IPv6 with external (public) connectivity |
| **Sub Interfaces** | [sub_interfaces](../sub_interfaces/sub_interfaces.ipynb) | Multiple virtual interfaces on a single dedicated NIC |